# Module 4 - Duplicate Data Handling

## 1. What are Duplicate Records?

Duplicate records are rows in a dataset that contain the same information as another row. They can occur due to repeated data entry, system errors, data merging, or repeated data imports.

Duplicate records can affect data analysis by increasing the number of observations and producing incorrect results.

In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv(r"C:\Users\HP\sprint-5-data-cleaning-preprocessing\data\hotel_bookings.csv")

# Display the first five records
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [2]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)
print("Dataset shape:", df.shape)

Number of duplicate records: 31994
Dataset shape: (119390, 32)


### Observation

The dataset was checked for duplicate records using all columns. The number of duplicate records was identified for further analysis and handling.

## 2. Exact Duplicates

Exact duplicates are rows where all column values are exactly the same as another row in the dataset.

These duplicate records contain the same information across every column and can usually be identified using `duplicated()`.

In [3]:
exact_duplicates = df[df.duplicated(keep=False)]

print("Number of exact duplicate records:", len(exact_duplicates))
print("\nExact duplicate records:")
print(exact_duplicates.head(10))

Number of exact duplicate records: 40165

Exact duplicate records:
            hotel  is_canceled  lead_time  arrival_date_year  \
4    Resort Hotel            0         14               2015   
5    Resort Hotel            0         14               2015   
21   Resort Hotel            0         72               2015   
22   Resort Hotel            0         72               2015   
39   Resort Hotel            0         70               2015   
43   Resort Hotel            0         70               2015   
132  Resort Hotel            1          5               2015   
138  Resort Hotel            1          5               2015   
198  Resort Hotel            0          0               2015   
200  Resort Hotel            0          0               2015   

    arrival_date_month  arrival_date_week_number  arrival_date_day_of_month  \
4                 July                        27                          1   
5                 July                        27                      

### Observation

The dataset was checked for exact duplicate records by comparing all columns. The identified duplicate rows contain the same values across the dataset and can be considered for removal after checking the data context.

## 3. Partial Duplicates

Partial duplicates are rows that match on selected key columns but may have different values in other columns.

They are more difficult to identify than exact duplicates because the entire row is not identical. For the hotel bookings dataset, columns such as `hotel`, `arrival_date_year`, `arrival_date_month`, `arrival_date_day_of_month`, and `adults` can be considered when checking for potential duplicate bookings.

In [4]:
partial_duplicates = df[df.duplicated(
    subset=['hotel', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'adults'],
    keep=False
)]

print("Number of potential partial duplicate records:", len(partial_duplicates))
print("\nPartial duplicate records:")
print(partial_duplicates.sort_values(
    ['hotel', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month']
).head(10))

Number of potential partial duplicate records: 118759

Partial duplicate records:
            hotel  is_canceled  lead_time  arrival_date_year  \
40560  City Hotel            0          0               2015   
40561  City Hotel            0        117               2015   
40562  City Hotel            0        117               2015   
40563  City Hotel            1        117               2015   
40564  City Hotel            1        117               2015   
40565  City Hotel            1        130               2015   
40566  City Hotel            0        117               2015   
40567  City Hotel            0        117               2015   
40568  City Hotel            1        117               2015   
40569  City Hotel            1        117               2015   

      arrival_date_month  arrival_date_week_number  arrival_date_day_of_month  \
40560             August                        31                          1   
40561             August                        31 

### Observation

Potential partial duplicates were identified by comparing selected booking-related columns. These records require further investigation because matching on selected columns does not necessarily mean that the bookings are true duplicates.

### 4.1 `duplicated()`

The `duplicated()` method identifies duplicate rows in a DataFrame. It returns `True` for rows that are duplicates of a previous row and `False` for unique rows.

In [5]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

print("\nDuplicate status:")
print(df.duplicated().value_counts())

Number of duplicate rows: 31994

Duplicate status:
False    87396
True     31994
Name: count, dtype: int64


### Observation

The `duplicated()` method was used to identify exact duplicate rows in the dataset. The number of duplicate and unique records was identified successfully.

### 4.2 `drop_duplicates()`

The `drop_duplicates()` method removes duplicate rows from a DataFrame and returns a cleaned DataFrame.

By default, it keeps the first occurrence of a duplicate row and removes the subsequent duplicate rows.

In [6]:
df_no_duplicates = df.drop_duplicates()

print("Original dataset shape:", df.shape)
print("Dataset shape after removing duplicates:", df_no_duplicates.shape)
print("Number of duplicate rows removed:", df.shape[0] - df_no_duplicates.shape[0])

Original dataset shape: (119390, 32)
Dataset shape after removing duplicates: (87396, 32)
Number of duplicate rows removed: 31994


### Observation

The `drop_duplicates()` method was used to remove exact duplicate records. The change in the number of rows shows how many duplicate records were removed from the dataset.

## 5. Duplicate Detection Based on Selected Columns

Instead of comparing all columns, duplicates can be identified using a subset of columns that represent important booking information.

For the hotel bookings dataset, selected booking-related columns can be used to identify potential duplicate records.

In [7]:
subset_cols = [
    'hotel',
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_day_of_month',
    'adults'
]

duplicate_count = df.duplicated(subset=subset_cols).sum()

print("Number of duplicates based on selected columns:", duplicate_count)

print("\nPotential duplicate records:")
print(df[df.duplicated(subset=subset_cols, keep=False)].head(10))

Number of duplicates based on selected columns: 114721

Potential duplicate records:
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   
5  Resort Hotel            0         14               2015               July   
6  Resort Hotel            0          0               2015               July   
7  Resort Hotel            0          9               2015               July   
8  Resort Hotel            1         85               2015               July   
9  Resort Hotel            1         75               2015               July   

   arrival_date_week_nu

### Observation

Duplicate records were identified using selected booking-related columns. This method can detect potential duplicates even when other columns in the records are different.

## 6. Handling Duplicate Records

Once duplicate records are identified, they can be handled in different ways depending on the data and business context.

Common approaches include removing duplicates, keeping the first or last occurrence, or reviewing potential duplicates before removal.

In [8]:
df_clean = df.drop_duplicates(
    subset=['hotel', 'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'adults'],
    keep='first'
)

print("Original dataset shape:", df.shape)
print("Dataset shape after handling duplicates:", df_clean.shape)
print("Records removed:", df.shape[0] - df_clean.shape[0])

Original dataset shape: (119390, 32)
Dataset shape after handling duplicates: (4669, 32)
Records removed: 114721


### Observation

Duplicate booking records were handled by keeping the first occurrence based on the selected booking-related columns. The number of records removed was compared with the original dataset.

## 7. Business Rules for Duplicate Removal

Duplicate removal should be based on business context rather than blindly deleting records.

Common rules include:

- Keep the record with more complete information.
- Keep the most relevant or recent booking record.
- Review records before removing potential duplicates.
- Do not remove records if they represent separate valid bookings.

In [9]:
df_business = df.copy()

df_business['missing_count'] = df_business.isnull().sum(axis=1)

df_business = df_business.sort_values(
    ['hotel', 'arrival_date_year', 'arrival_date_month',
     'arrival_date_day_of_month', 'adults', 'missing_count']
)

df_business_clean = df_business.drop_duplicates(
    subset=['hotel', 'arrival_date_year', 'arrival_date_month',
            'arrival_date_day_of_month', 'adults'],
    keep='first'
)

df_business_clean = df_business_clean.drop(columns=['missing_count'])

print("Original dataset shape:", df.shape)
print("Dataset shape after applying business rules:", df_business_clean.shape)

Original dataset shape: (119390, 32)
Dataset shape after applying business rules: (4669, 32)


### Observation

Business rules were applied to retain the more complete record when potential duplicates were found. This approach helps avoid blindly removing useful information from the dataset.

## 8. Why Blindly Deleting Duplicates Can Be Dangerous

Removing duplicate records without understanding the business context can lead to incorrect results.

- Some similar records may represent valid and separate hotel bookings.
- Duplicate detection based on the wrong columns can remove valid records.
- Using `keep='first'` or `keep='last'` without checking the data may keep an incorrect record.
- Removing records can affect booking counts and other analysis results.
- Important information may be lost if duplicate-looking records represent different transactions.

The safe approach is to understand why the records are duplicated, define suitable business rules, and then decide whether to remove, retain, or review them.

In [10]:
id_columns = [
    'hotel',
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_day_of_month',
    'adults'
]

potential_duplicates = df[df.duplicated(
    subset=id_columns,
    keep=False
)]

print("Potential duplicate records:", len(potential_duplicates))
print("\nThese records should be reviewed before removal.")

Potential duplicate records: 118759

These records should be reviewed before removal.


### Observation

Duplicate records should not be removed blindly. Potential duplicates must be reviewed based on the booking details and business rules to avoid losing valid hotel booking records.